# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Read `skills/README.md`, then loaded `skills/writing-research-papers/SKILL.md` and `skills/deploying-static-pages/SKILL.md` as directed on this assignment's card. This notebook mirrors the deployed paper at https://m-husnain-dev.github.io/ml-internship/paper/ — same question, same numbers, reproducible end to end from this repo's starter dataset.*

## 1. Question

*The research question and the decision it supports.*

**Question:** Which declining content pages should a reviewer with limited time check first for a possible refresh — and can a learned model actually outperform a simple hand-written rule at that task?

**Lane:** Refresh / Content Opportunity Scoring (Lane 2).

**The decision this supports:** a content team with a large page inventory can't manually review every page. This project produces a ranked review queue so a human reviewer spends limited time where it's most likely to matter, with a reason attached to every recommendation — not a verdict, and not an automated action.

**Why this earns a full research paper and not just a working script:** the honest answer turned out to be more interesting than the naive one — the first, natural way to evaluate the baseline was accidentally circular, and once fixed, the baseline beat both learned models. That tension (expected: model wins → actual: baseline wins, and here's exactly why) is the paper's whole thread.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

print(f"Total rows in starter dataset: {len(df)}")
print(f"Eligible rows (impressions_90d > 0, content_age_days >= 90): {len(eligible)}")
print(f"Unique pseudonymized clients: {eligible['client_id'].nunique()}")

**Release:** FlyRank ML Internship starter dataset, `data/raw/content_refresh_anonymized.csv` — 30,000 rows, one row per pseudonymized content item, 32 pseudonymized clients, trailing-90-day search and engagement metrics.

**Date window:** trailing 90-day snapshot per row (no absolute dates in the starter release).

**Excluded, and why:**
- No client names, domains, URLs, or raw queries — never present in this release to begin with; the dataset ships pseudonymized by design.
- `trend_direction` / `trend_pct` excluded from every model **feature** set — used only to construct labels and evaluation targets. Using either as a feature would let a model learn to copy its own answer (see Methodology).
- No product-decision flags (`health_score`, `priority_score`, etc.) — none exist in this dataset, confirmed directly rather than assumed.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Starter label:** `is_declining_label` (`trend_direction == "down"`) — a proxy from the current window, not a validated future-window outcome. This limitation is carried through the whole project and named explicitly in Section 5.

**Stronger evaluation target, `true_priority`:** built after discovering the starter label made the baseline's own evaluation circular (Section 4). Flags pages that are both severely declining (bottom third of decline magnitude among decliners) and carry real traffic at stake (impressions at or above the median for declining pages) — built only from label-construction fields, never used as a feature.

In [ ]:
decliners = eligible[eligible["is_declining_label"] == 1]
severe_cutoff = decliners["trend_pct"].quantile(1/3)
impr_cutoff = decliners["impressions_90d"].median()
eligible["true_priority"] = ((eligible["is_declining_label"] == 1) &
                               (eligible["trend_pct"] <= severe_cutoff) &
                               (eligible["impressions_90d"] >= impr_cutoff)).astype(int)

print(f"true_priority rate (all eligible): {eligible['true_priority'].mean():.3f}")

**Baseline (plain words):** a page is worth reviewing if it's declining and visible (real impressions), ranked higher the more impressions it's wasting, with extra priority if it's also either moderately stale (30–110 days since last update) or badly underperforming CTR for its position.

In [ ]:
declining_flag = (eligible["trend_direction"] == "down").astype(int)
visible_flag = (eligible["impressions_90d"] >= 500).astype(int)
stale_flag = ((eligible["days_since_last_update"] >= 30) & (eligible["days_since_last_update"] < 110)).astype(int)
low_ctr_flag = ((eligible["ctr"] < 0.5) & (eligible["avg_position"] > 0) & (eligible["avg_position"] <= 20)).astype(int)

eligible["baseline_score"] = declining_flag * visible_flag * eligible["impressions_90d"] * (1 + 0.5*stale_flag + 0.5*low_ctr_flag)
print(f"Flagged by baseline: {(eligible['baseline_score']>0).sum()} of {len(eligible)}")

**Features (11, all pre-decision, none label-derived):** `impressions_90d`, `sessions_90d`, `avg_position`, `ctr`, `word_count`, `content_age_days`, `days_since_last_update`, `search_volume`, `cpc`, `engagement_rate`, `scroll_rate`.

**Models:** Logistic Regression, then Random Forest (200 trees, max depth 8, min leaf 20) — readable first, stronger second, per the "simplicity is a feature" rule.

**Validation design:** the starter data spans 32 clients. A plain random split lets rows from the same client appear in both train and test. All headline results use a **client-grouped split** (30% test, zero client overlap verified); a random split is also reported once, specifically to show the size of the gap.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

feature_cols = ["impressions_90d", "sessions_90d", "avg_position", "ctr", "word_count",
                 "content_age_days", "days_since_last_update", "search_volume", "cpc",
                 "engagement_rate", "scroll_rate"]

X = eligible[feature_cols].fillna(0)
y_priority = eligible["true_priority"]
groups = eligible["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y_priority, groups))

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Client overlap between train/test: {len(train_clients & test_clients)}")

**Leakage check:** deliberately added `trend_pct` back as a feature to confirm the test harness catches a planted leak.

In [ ]:
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y_priority.iloc[train_idx], y_priority.iloc[test_idx]

rf_honest = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_honest.fit(X_train_g, y_train_g)
honest_auc = roc_auc_score(y_test_g, rf_honest.predict_proba(X_test_g)[:,1])

X_leaky = eligible[feature_cols + ["trend_pct"]].fillna(0)
rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_leaky.fit(X_leaky.iloc[train_idx], y_train_g)
leaky_auc = roc_auc_score(y_test_g, rf_leaky.predict_proba(X_leaky.iloc[test_idx])[:,1])

print(f"Honest AUC (no trend_pct): {honest_auc:.3f}")
print(f"Leaky AUC (WITH trend_pct): {leaky_auc:.3f}  <- confirms the harness catches a planted leak")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Circularity finding: baseline evaluated against its own gating label
print("--- Circularity check ---")
print(f"Baseline P@20 vs is_declining_label (the label it's gated on): {precision_at_k(eligible['baseline_score'].iloc[test_idx].values, eligible['is_declining_label'].iloc[test_idx].values, 20):.3f}  <- trivially perfect, not a real result")

In [ ]:
baseline_test = eligible["baseline_score"].iloc[test_idx].values

scaler_cols = feature_cols
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_g, y_train_g)
lr_scores = lr.predict_proba(X_test_g)[:,1]

rf_scores = rf_honest.predict_proba(X_test_g)[:,1]

print("--- Honest comparison (true_priority target, client-grouped split) ---")
print(f"{'Method':<20}{'P@20':<10}{'P@50':<10}{'ROC-AUC'}")
print(f"{'Baseline rule':<20}{precision_at_k(baseline_test, y_test_g.values, 20):<10.3f}{precision_at_k(baseline_test, y_test_g.values, 50):<10.3f}{roc_auc_score(y_test_g, baseline_test):.3f}")
print(f"{'Logistic Reg':<20}{precision_at_k(lr_scores, y_test_g.values, 20):<10.3f}{precision_at_k(lr_scores, y_test_g.values, 50):<10.3f}{roc_auc_score(y_test_g, lr_scores):.3f}")
print(f"{'Random Forest':<20}{precision_at_k(rf_scores, y_test_g.values, 20):<10.3f}{precision_at_k(rf_scores, y_test_g.values, 50):<10.3f}{roc_auc_score(y_test_g, rf_scores):.3f}")
print(f"\nBase rate (true_priority, test set): {y_test_g.mean():.3f}")

**The honest finding:** against the circular label, the baseline scores a trivial 1.000. Against `true_priority`, the baseline (ROC-AUC 0.866) clearly beats both Logistic Regression (0.603) and Random Forest (0.598). The models were trained on `is_declining_label`, not `true_priority` — a target mismatch that looks like the real driver of the gap, not model capacity.

## 5. Limitations

*What this work cannot claim.*

- **Observational, cross-sectional** — no experiment was run, so nothing here supports a causal claim that refreshing a page *will* improve it. The honest framing is decision-support: these pages *look worth reviewing first*.
- **Starter dataset only** (30,000 rows, 32 clients) — not the full warehouse. Not yet validated at scale.
- `is_declining_label` **is a proxy**, not a genuine future-window outcome — a stronger version would define `prior window → future window` labels on the full warehouse with a complete leakage audit.
- The **stale (110d+) bucket is small** (n=230) and is never used as standalone evidence.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
def reason_code(low_ctr, stale):
    return "low_ctr_visible_page" if low_ctr else ("moderately_stale_page" if stale else "declining_with_demand")

eligible["reason_code"] = [reason_code(l, s) for l, s in zip(low_ctr_flag, stale_flag)]
ARCHETYPE_ACTION = {
    "low_ctr_visible_page": "rewrite_title_meta",
    "moderately_stale_page": "content_refresh_review",
    "declining_with_demand": "monitor_or_review",
}
eligible["action"] = eligible["reason_code"].map(ARCHETYPE_ACTION)
eligible.loc[eligible["baseline_score"] == 0, "action"] = "monitor"

ranked = eligible.sort_values("baseline_score", ascending=False).reset_index(drop=True)
print(ranked.loc[ranked['baseline_score']>0, 'reason_code'].value_counts())
ranked[["content_id","baseline_score","reason_code","action"]].head(10)

Because the baseline outperformed both models under honest evaluation, the shipped playbook is built on the **baseline rule**, not the more complex model. Reason codes map to concrete actions: `low_ctr_visible_page` → rewrite title/meta, `moderately_stale_page` → content refresh review, `declining_with_demand` → monitor or review. **Never automated:** auto-publishing, auto-consolidating, or presenting this as proof a refresh will work — always a human-reviewed queue.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
import os
os.makedirs("../figures", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7,4))
methods = ["Baseline rule", "Logistic Regression", "Random Forest"]
aucs = [roc_auc_score(y_test_g, baseline_test), roc_auc_score(y_test_g, lr_scores), roc_auc_score(y_test_g, rf_scores)]
ax.bar(methods, aucs, color=["#3B5BDB","#FF6B4A","#FF6B4A"])
ax.axhline(0.5, color="#999999", linestyle="--", label="random chance")
ax.set_ylabel("ROC-AUC (vs true_priority)")
ax.set_title("Honest comparison: baseline vs. models")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/capstone_honest_comparison.png", dpi=150)
plt.close()
print("Saved work/figures/capstone_honest_comparison.png")

In [ ]:
import json

metrics = {
    "eligible_rows": int(len(eligible)),
    "n_clients": int(eligible["client_id"].nunique()),
    "baseline_auc_true_priority": float(roc_auc_score(y_test_g, baseline_test)),
    "logistic_auc_true_priority": float(roc_auc_score(y_test_g, lr_scores)),
    "random_forest_auc_true_priority": float(roc_auc_score(y_test_g, rf_scores)),
    "leakage_check": {"honest_auc": float(honest_auc), "leaky_auc": float(leaky_auc)},
}
with open("../outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))
print("\nSaved work/outputs/capstone_metrics.json — the receipts behind the deployed paper's numbers.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself in Colab**; all numbers reproduce exactly against this repo's real starter dataset with fixed seeds, and match the deployed paper
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.